<a href="https://colab.research.google.com/github/cyberwave-os/cyberwave-sdk-python/blob/main/examples/notebooks/hand_eye_calibration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Hand-eye calibration with the Cyberwave SDK

A wrist-mounted camera only tells you where things are *relative to the camera*. To act on
what it sees, you need where the camera sits **on the arm** — the transform from the wrist
link to the camera's optical frame. Typing that in from a CAD drawing is usually wrong by
millimetres and a degree or two, and the error shows up as a robot that misses.

Hand-eye calibration measures it instead. You hold a printed board still, move the wrist to
a dozen orientations, and solve `AX = XB` for `X`.

**This notebook runs end to end with no robot and no camera.** Colab has neither, so we
build a synthetic arm whose answer we already know, recover it, and check we got it back.
That makes every number here verifiable. The final section shows the real-hardware code,
which is the same API with the fake parts removed.

---


## 1. Install

The calibration extra pulls in OpenCV, which is where `cv2.aruco` and `cv2.calibrateHandEye`
live. Both are main-module OpenCV since 4.7 — no contrib build needed.


In [ ]:
%pip install -q 'cyberwave[calibration]'

import cv2, numpy as np
print('opencv', cv2.__version__)
assert tuple(int(p) for p in cv2.__version__.split('.')[:2]) >= (4, 7), (
    'Need OpenCV >= 4.7 for the ArUco API this uses'
)
print('aruco available:', hasattr(cv2, 'aruco'))


## 2. The frame convention

Every transform here is a 4x4 named `a_to_b`, meaning *the pose of b expressed in a*. Compose
them left to right: `base_to_gripper @ gripper_to_camera` is the camera in base coordinates.

The four frames that matter:

| Frame | What it is |
|---|---|
| `base` | The arm's base. Fixed. |
| `gripper` | The link the camera is bolted to (`fk_frame`). Moves with the arm. |
| `camera` | The camera's **optical** frame: +Z along the view axis, +X right, +Y down. |
| `target` | The calibration board. Must not move during a session. |

`gripper_to_camera` is the unknown — call it `X`. It is constant, because the camera is bolted
to the wrist.


In [ ]:
from cyberwave.calibration import make_transform, invert, describe_pose

def rot(axis: str, deg: float) -> np.ndarray:
    """A 3x3 rotation about a principal axis. Only used to build the fake arm."""
    t = np.radians(deg)
    c, s = np.cos(t), np.sin(t)
    return {
        'x': np.array([[1, 0, 0], [0, c, -s], [0, s, c]]),
        'y': np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]]),
        'z': np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]]),
    }[axis]

# The ground truth we are pretending not to know: a camera 2.1 cm right, 3.4 cm back
# and 5.2 cm forward of the wrist link, tilted off-axis so a solver that only
# recovers translation cannot pass by accident.
TRUE_X = make_transform(rot('x', 18) @ rot('y', -9), [0.021, -0.034, 0.052])
print('ground truth gripper_to_camera:')
print(describe_pose(TRUE_X))


## 3. Build a synthetic session

On real hardware each sample is a pair: the arm's pose from forward kinematics, and the
board's pose from `solvePnP` on a camera frame. Here we generate both from `TRUE_X`, so the
data is exactly consistent and any error we see is the solver's, not the setup's.

**The one thing that matters for a good calibration: rotate about genuinely different axes.**
The maths recovers translation only from rotation, so poses that merely slide the wrist
around leave the problem ill-posed. Notice the loop below tilts about x, y *and* z.


In [ ]:
# The board sits on the table, still, for the whole session.
BASE_TO_TARGET = make_transform(rot('x', 90), [0.28, -0.05, 0.11])

# A dozen wrist poses, each rotated about a different combination of axes.
base_to_gripper = [
    make_transform(
        rot('x', 14 * i) @ rot('y', 8 * i) @ rot('z', 5 * i),
        [0.20 + 0.010 * i, 0.010 * i, 0.30],
    )
    for i in range(12)
]

# What the camera would have seen from each pose, given the truth.
camera_to_target = [invert(g @ TRUE_X) @ BASE_TO_TARGET for g in base_to_gripper]

print(f'{len(base_to_gripper)} samples')


## 4. Solve

`solve_hand_eye` is the low-level entry point: two matched pose lists in, one transform out.
(On real hardware you use `HandEyeSession`, which assembles these lists itself — see §8 for
why that matters.)


In [ ]:
from cyberwave.calibration import solve_hand_eye, DEFAULT_METHOD

result = solve_hand_eye(base_to_gripper, camera_to_target)

error_m = np.linalg.norm(result.gripper_to_camera[:3, 3] - TRUE_X[:3, 3])
print(f'method              : {result.method}  (default: {DEFAULT_METHOD})')
print(f'samples             : {result.sample_count}')
print(f'recovered position  : {result.gripper_to_camera[:3, 3]}')
print(f'true position       : {TRUE_X[:3, 3]}')
print(f'error vs truth      : {error_m:.3e} m   <- machine precision, as it should be')


## 5. Reading the quality numbers

On real data you have no ground truth, so you judge the answer by two very different
statistics. **They are not interchangeable, and when they disagree the second one wins.**

### The residual — do the captures agree with each other?

The board did not move, so `base_to_gripper @ X @ camera_to_target` should give the same board
pose for every sample. How much it varies is the residual.


In [ ]:
print(f'residual_translation_m     : {result.residual_translation_m:.3e} m')
print(f'max_residual_translation_m : {result.max_residual_translation_m:.3e} m')
print(f'residual_rotation_deg      : {result.residual_rotation_deg:.3e} deg')
print()
print('relative (AX=XB) fit error — how well X reconciles the motions observed:')
print(f'  relative_translation_m     : {result.relative_translation_m:.3e} m')
print(f'  relative_rotation_deg      : {result.relative_rotation_deg:.3e} deg')
print(f'  relative_translation_ratio : {result.relative_translation_ratio:.3e}  (dimensionless)')


### Leave-one-out stability — will this number hold?

This is the one to trust. It re-solves with each sample dropped in turn and reports how far
the answer moves. That asks the question you actually have, where the residual only asks
whether your captures are self-consistent.

They come apart in practice. Measured on an SO-101 wrist camera, runs whose mount position
was reproducible across independent sessions landed at **1.9–3.5 mm** stability; ill-conditioned
ones reached **7.7–24 mm** — almost entirely along the camera's optical axis — while their
residuals still looked respectable.

Needs at least `MIN_LEAVE_ONE_OUT_SAMPLES` samples; below that it returns `None`, which means
*"cannot say"*, not *"fine"*.


In [ ]:
from cyberwave.calibration import (
    leave_one_out_stability,
    stability_from_result,
    MIN_LEAVE_ONE_OUT_SAMPLES,
    MIN_SAMPLES,
)

print(f'MIN_SAMPLES={MIN_SAMPLES}  MIN_LEAVE_ONE_OUT_SAMPLES={MIN_LEAVE_ONE_OUT_SAMPLES}')

loo = leave_one_out_stability(base_to_gripper, camera_to_target)
if loo is None:
    print('not enough samples to say')
else:
    print(f'max_stddev_m       : {loo.max_stddev_m:.3e} m')
    print(f'worst_sample_index : {loo.worst_sample_index}')
    print(f'subset_count       : {loo.subset_count}')


### Turning that into a verdict

`verdict()` folds both numbers into a label, keyed on stability with the residual as a
secondary check — the same label the dashboard shows, so a notebook and the UI agree.

It reads exactly three thresholds off a config object. **The SDK ships no defaults for them,
deliberately:** a stability bar measured on one arm is not a default for another, and a
library default would be a measurement nobody took presented as one they did. The values
below are the ones measured on the SO-101 — substitute your own once you have them.


In [ ]:
from dataclasses import dataclass
from cyberwave.calibration import verdict

@dataclass(frozen=True)
class Thresholds:
    """The only three fields `verdict()` reads. Measured on an SO-101.

    The full `HandEyeFlowConfig` also carries twin UUIDs, a kinematics object and
    a board — everything a live capture flow needs. For judging a result you
    already have, these three are enough.
    """
    good_stability_m: float = 0.004   # at or under -> 'good'
    bad_stability_m: float = 0.008    # over       -> 'bad'
    good_residual_m: float = 0.005    # secondary check only

thresholds = Thresholds()

validated = solve_hand_eye(base_to_gripper, camera_to_target).with_validation(
    leave_one_out=loo,
)
stability_m = stability_from_result(validated)

print('stability :', 'n/a' if stability_m is None else f'{stability_m:.3e} m')
print('verdict   :', verdict(validated.residual_translation_m, stability_m, config=thresholds))


## 6. What a *bad* calibration looks like

This is the section worth internalising.

Below we sweep one variable: how much the wrist tilts about a **second** axis between captures.
Everything else is identical. As that secondary tilt shrinks, the problem becomes progressively
ill-posed — the classic mistake of sliding the wrist around instead of genuinely reorienting it.

Watch the three columns. The residual does not move at all.


In [ ]:
from cyberwave.calibration import HandEyeDegenerateError

print(f"{'2nd-axis tilt':>14} | {'residual':>9} | {'stability':>10} | {'TRUE error':>11} | verdict")
print("-" * 68)

for wobble_deg in (0.15, 0.25, 0.40, 1.0, 3.0):
    poses = [
        make_transform(rot('x', 12 * i) @ rot('y', wobble_deg * i), [0.20 + 0.01 * i, 0.0, 0.30])
        for i in range(12)
    ]
    views = [invert(g @ TRUE_X) @ BASE_TO_TARGET for g in poses]

    # 0.5 mm of board-detection noise, so this behaves like real data.
    rng = np.random.default_rng(0)
    noisy = []
    for v in views:
        p = v.copy()
        p[:3, 3] += rng.normal(0.0, 0.0005, 3)
        noisy.append(p)

    try:
        r = solve_hand_eye(poses, noisy)
    except HandEyeDegenerateError:
        print(f"{wobble_deg:>11.2f}deg | refused outright - no transform is determined")
        continue

    loo = leave_one_out_stability(poses, noisy)
    stab = None if loo is None else loo.max_stddev_m
    true_error = np.linalg.norm(r.gripper_to_camera[:3, 3] - TRUE_X[:3, 3])

    print(
        f"{wobble_deg:>11.2f}deg | "
        f"{r.residual_translation_m * 1000:6.2f} mm | "
        f"{'n/a' if stab is None else format(stab * 1000, '7.2f') + ' mm'} | "
        f"{true_error * 1000:8.2f} mm | "
        f"{verdict(r.residual_translation_m, stab, config=thresholds)}"
    )


Look at the residual column: **it is the same value on every row**, while the true error varies
by roughly 6x and the verdict flips from `marginal` to `good`. The residual only knows whether
your captures agree with each other, and captures taken from near-identical orientations agree
beautifully while pinning down almost nothing.

Stability tracks the true error closely, which is exactly why the verdict is keyed on it.

Push the secondary tilt low enough and the SDK stops guessing altogether and raises
`HandEyeDegenerateError` rather than returning a confident wrong answer.

<div class="alert alert-warning">

**Neither number is an error bar.** A consistent mistake — forward kinematics reported about the
wrong link, say — produces a *small* residual **and** good stability, because the resulting system
is still perfectly self-consistent. It just solves for the wrong thing. That class of error cannot
be detected after the fact; it has to be prevented by construction, which is what the next section
is about.

</div>


## 7. Print a board

The board is a physical input, and two things about it must be exactly right: the `dictionary`
has to match what you print, and `square_size_m` has to match the print **after** your printer's
scaling. Measure the printed squares with calipers and put that number in — do not trust the
value you asked for.


In [ ]:
from cyberwave.calibration import CharucoBoard

board = CharucoBoard(
    squares=(11, 8),          # (columns, rows) of TOTAL squares - order matters
    square_size_m=0.020,      # measure the print!
    marker_size_m=0.015,      # must be smaller than square_size_m
    dictionary='DICT_5X5_1000',
)

image = board.generate_image(pixels_per_square=120)
cv2.imwrite('charuco_board.png', image)
print('board metadata:', board.to_metadata())

from IPython.display import Image, display
display(Image('charuco_board.png', width=460))


<div class="alert alert-warning">

**`squares` is `(columns, rows)`, and getting it backwards fails silently.** OpenCV's
`detectBoard` interpolates *zero* chessboard corners from a transposed grid even when every
marker is found with a valid id — so it looks exactly like "the board isn't in frame" rather
than a configuration error.

</div>


## 8. On real hardware

Everything above used `solve_hand_eye` directly, which takes two pose lists. On real hardware
use `HandEyeSession` instead — it **assembles both lists itself**, and that is a correctness
feature, not convenience: swapping the two lists yields a system that is still exactly
consistent, solves for the inverse board pose, and reports a machine-zero residual. There is
no number that catches it. The session also validates that the camera twin really is docked to
`fk_frame`, ruling out the other silent failure.

The code below is illustrative — it needs a real arm, camera and board, so it will not run here.


In [ ]:
# NOT RUNNABLE IN COLAB - needs real hardware. Shown for reference.
REAL_HARDWARE = False

if REAL_HARDWARE:
    from cyberwave import Cyberwave
    from cyberwave.calibration import HandEyeSession, CharucoBoard

    client = Cyberwave()                      # reads CYBERWAVE_API_KEY
    arm = client.twin(twin_id='<arm-twin-uuid>')
    camera = client.twin(twin_id='<camera-twin-uuid>')

    session = HandEyeSession(
        camera_twin=camera,
        arm_twin=arm,
        fk_frame='gripper',   # the link the camera is bolted to
        board=board,
        intrinsics=None,      # omit -> solved from your own captures at solve()
    )

    # Hand-guide the wrist to a new orientation, let it settle, capture. Repeat.
    # Tilt about genuinely different axes; vary the distance to the board too.
    for _ in range(12):
        input('Move the wrist to a new orientation, then press Enter...')
        try:
            session.add_sample()          # captures a frame + reads joints for you
            print(f'  captured ({session.sample_count} usable)')
        except Exception as exc:
            print(f'  rejected: {exc}')

    result = session.solve()              # fits intrinsics, then solves hand-eye

    stability = stability_from_result(result)
    print('verdict:', verdict(result.residual_translation_m, stability, config=thresholds))

    # Write it to the camera twin's docking offset, with provenance.
    camera.calibration.set(
        result,
        board=board,
        intrinsics=session.intrinsics,
        fk_frame='gripper',
    )


### Intrinsics

Intrinsics are an **input** the hand-eye solve cannot recover from: every board pose inherits
their error, and it surfaces only as an inflated residual with nothing pointing at the cause.

Pass `intrinsics=None` and the session fits them from your own captures at `solve()` time —
finding board corners needs no camera model, only turning them into a pose does. That is
strictly better than a field-of-view estimate, which is good enough to draw a frustum and not
good enough for metric work.


In [ ]:
# Check what was fitted, when you let the session solve them:
#
#   result.solved_intrinsics.rms_reprojection_px        # under ~1 px is good
#   result.solved_intrinsics.principal_point_is_centred # False usually means a
#                                                       # wrong board spec, not a
#                                                       # genuinely offset sensor
#
# You can also fit them standalone from a set of board images:
from cyberwave.calibration import solve_intrinsics, MIN_INTRINSICS_VIEWS
print(f'solve_intrinsics(images, board) needs >= {MIN_INTRINSICS_VIEWS} views')


## 9. Checklist

When a result looks wrong, work down this list — it is ordered by how often each one is the
actual cause:

1. **Rotation variety.** Tilt about genuinely different axes, not just one. This is the single
   biggest factor, and its failure mode is a *small* residual with a wrong answer.
2. **Board spec.** `square_size_m` measured off the print with calipers; `dictionary` matching
   what you actually printed; `squares` as `(columns, rows)`.
3. **The board did not move.** Clamp it. A bumped board mid-session corrupts every sample after.
4. **`fk_frame` is the link the camera is really bolted to.** Wrong link = wrong answer, good
   residual.
5. **Distance variety.** Vary how far the wrist is from the board, not just its orientation.
6. **Judge by stability, not the residual.** Under ~4 mm is good; over ~8 mm do not apply it.

### Further reading

- `HandEyeSession` — the guided capture/solve API
- `camera.calibration.set(...)` — persist the result to the twin's docking offset
- Docs: <https://docs.cyberwave.com/feature-reference/hand-eye-calibration>
